In [1]:
import os
import json
import torch
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from model import GrokkingSetup
import plotly.express as px

# 64 Embedding Dimensions

In [2]:
def GetModelFromConfig(path):
    config_path = os.path.join(path, "config.json")
    with open(config_path, "r") as f:
        config = json.load(f)
        
    model = GrokkingSetup(
        d_model=config["d_model"], 
        voc_size=config["voc_size"],
        pos_encoding=config.get("pos_encoding", 4), 
        d_k=config["d_k"], 
        d_v=config["d_v"], 
        n_layers=config["n_layers"]
    )
    
    return model

In [6]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import numpy as np

def plot_input_only_plotly(model, N):
    weights = model.tokens_embed.weight.detach().cpu().numpy()

    offset = 2 * N * N 
    labels = np.arange(-N, N + 1)
    indices = labels + offset
    
    valid_indices = [i for i in indices if 0 <= i < weights.shape[0]]
    input_weights = weights[valid_indices]
    labels = labels[:len(valid_indices)]
    
    pca = PCA(n_components=2)
    embeds_2d = pca.fit_transform(input_weights)
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=embeds_2d[:, 0],
        y=embeds_2d[:, 1],
        mode='markers+text',
        text=[str(l) for l in labels],
        textposition="top center",
        marker=dict(
            size=14,
            color='red',
            line=dict(width=1, color='black')
        ),
        hoverinfo='text',
        hovertext=[f"Wartość wejściowa: {l}<br>Index w embed: {l + offset}" for l in labels]
    ))
    
    fig.update_layout(
        title=f"Struktura liczb wejściowych (Embeddings) dla N={N}",
        xaxis_title="PCA Component 1",
        yaxis_title="PCA Component 2",
        plot_bgcolor='white',
        width=700,
        height=700
    )
    
    fig.update_xaxes(showgrid=True, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridcolor='lightgray')
    
    fig.show()

In [7]:
path = r"experiments/run_L2_D32_20260522_151145"

model = GetModelFromConfig(path)

model_path = r"experiments/run_L2_D32_20260522_151145/checkpoints/model_epoch_5000.pt"

model.load_state_dict(torch.load(model_path))

plot_input_only_plotly(model, 6)

In [8]:
def plot_input_pca_3d_plotly(model, N):
    weights = model.tokens_embed.weight.detach().cpu().numpy()

    input_offset = 2 * N * N   
    input_values = list(range(-N, N + 1))
    input_indices = [val + input_offset for val in input_values]
    
    if max(input_indices) >= weights.shape[0] or min(input_indices) < 0:
        return

    active_weights = weights[input_indices]
    
    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(active_weights)
    
    df = pd.DataFrame(embeds_3d, columns=['PCA1', 'PCA2', 'PCA3'])
    df['label'] = input_values

    fig = px.scatter_3d(
        df, 
        x='PCA1', 
        y='PCA2', 
        z='PCA3',
        color='label',
        text='label',
        color_continuous_scale='Portland',
        title=f"3D PCA: Input Tokens Structure (N={N}, offset={input_offset})",
        labels={'label': 'Wartość wejściowa'},
        opacity=0.8
    )

    fig.update_traces(
        marker=dict(size=6, line=dict(width=1, color='Black')),
        textposition='top center'
    )

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis_title='PCA 1',
            yaxis_title='PCA 2',
            zaxis_title='PCA 3'
        )
    )
    
    fig.show()


In [9]:
plot_input_pca_3d_plotly(model, N=6)

In [12]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

def plot_real_output_plotly(model, N):
    weights = model.unembed_re.weight.detach().cpu().numpy()
    
    output_offset = 2 * N * N
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < weights.shape[0]:
            active_indices.append(idx)
            labels.append(val)

    active_weights = weights[active_indices]


    weight_norms = np.linalg.norm(active_weights, axis=1)
    valid_mask = weight_norms > 0.4
    
    active_weights = active_weights[valid_mask]
    labels = np.array(labels)[valid_mask]

    pca = PCA(n_components=2)
    embeds_2d = pca.fit_transform(active_weights)
    
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=embeds_2d[:, 0],
        y=embeds_2d[:, 1],
        mode='markers+text',
        text=[str(l) for l in labels],
        textposition="top right",
        textfont=dict(size=10, family="Arial Black"),
        marker=dict(
            size=12,
            color=labels,
            colorscale='RdBu',
            showscale=True,
            colorbar=dict(title="Wartość Real"),
            line=dict(width=1, color='Black')
        ),
        hoverinfo='text',
        hovertext=[f"Token ID: {l + output_offset}<br>Wartość Real: {l}" for l in labels]
    ))

    fig.update_layout(
        title=dict(
            text=f"Interaktywna struktura wyjść (Unembed Real) dla N={N}",
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(title="PCA Component 1", gridcolor='rgba(200,200,200,0.2)'),
        yaxis=dict(title="PCA Component 2", gridcolor='rgba(200,200,200,0.2)'),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=900,
        height=800,
        showlegend=False
    )
    
    fig.show()
    plt.show()

In [13]:
plot_real_output_plotly(model, N=6)

In [14]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import numpy as np

def plot_imag_output_plotly(model, N=5):
    weights = model.unembed_im.weight.detach().cpu().numpy()
    
    output_offset = 2 * N * N
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < weights.shape[0]:
            active_indices.append(idx)
            labels.append(val)
            
    active_weights = np.array(weights[active_indices])
    labels = np.array(labels)

    weight_norms = np.linalg.norm(active_weights, axis=1)
    valid_mask = weight_norms > 0.4
    
    filtered_weights = active_weights[valid_mask]
    filtered_labels = labels[valid_mask]
    
    pca = PCA(n_components=2)
    embeds_2d = pca.fit_transform(filtered_weights)
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=embeds_2d[:, 0],
        y=embeds_2d[:, 1],
        mode='markers+text',
        text=[str(l) for l in filtered_labels],
        textposition="top center",
        marker=dict(
            size=12,
            color=filtered_labels,
            colorscale='rdbu_r',
            showscale=True,
            colorbar=dict(title="Wartość Imaginary"),
            line=dict(width=1, color='Black')
        ),
        hoverinfo='text',
        hovertext=[f"Wartość Imag: {l}" for l in filtered_labels]
    ))
    
    fig.update_layout(
        title=f"Struktura wyjść modelu (Unembed Imaginary) dla N={N}",
        xaxis_title="PCA Component 1",
        yaxis_title="PCA Component 2",
        plot_bgcolor='white',
        width=900,
        height=800
    )
    
    fig.show()

In [15]:
plot_imag_output_plotly(model, N=6)

In [17]:
def plot_real_output_pca_3d(model, N):
    output_offset = 2 * N * N
    weights = model.unembed_re.weight.detach().cpu().numpy()
    
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < weights.shape[0]:
            active_indices.append(idx)
            labels.append(val)
            
    active_weights = weights[active_indices]
    

    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(active_weights)
    
    df = pd.DataFrame(embeds_3d, columns=['PCA1', 'PCA2', 'PCA3'])
    df['label'] = labels
    df['abs_val'] = df['label'].abs()

    fig = px.scatter_3d(
        df, 
        x='PCA1', 
        y='PCA2', 
        z='PCA3',
        color='label',
        text='label',
        color_continuous_scale='Spectral',
        title=f"3D PCA: Struktura Unembed Real (N={N}, offset={output_offset})",
        labels={'label': 'Wartość'},
        opacity=0.8
    )


    fig.update_traces(
        marker=dict(size=5, line=dict(width=1, color='Black')),
        textposition='top center'
    )

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis_title='PCA 1',
            yaxis_title='PCA 2',
            zaxis_title='PCA 3'
        )
    )
    
    fig.show()

In [18]:
plot_real_output_pca_3d(model, N=6)

In [19]:
def plot_output_im_pca_3d_plotly(model, N):
    weights = model.unembed_im.weight.detach().cpu().numpy()
    
    output_offset = 2 * N * N
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < weights.shape[0]:
            active_indices.append(idx)
            labels.append(val)
            
    active_weights = weights[active_indices]

    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(active_weights)
    
    df = pd.DataFrame(embeds_3d, columns=['PCA1', 'PCA2', 'PCA3'])
    df['label'] = labels

    fig = px.scatter_3d(
        df, 
        x='PCA1', 
        y='PCA2', 
        z='PCA3',
        color='label',
        text='label',
        color_continuous_scale='Viridis',
        title=f"3D PCA: Struktura Unembed Imaginary (N={N}, offset={output_offset})",
        labels={'label': 'Wartość urojona'},
        opacity=0.8
    )

    fig.update_traces(
        marker=dict(size=5, line=dict(width=1, color='Black')),
        textposition='top center'
    )

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis_title='PCA 1',
            yaxis_title='PCA 2',
            zaxis_title='PCA 3'
        )
    )
    
    fig.show()

In [20]:
plot_output_im_pca_3d_plotly(model, N=6)

In [22]:
def plot_joint_complex_pca_3d(model, N=6):
    w_re = model.unembed_re.weight.detach().cpu().numpy()
    w_im = model.unembed_im.weight.detach().cpu().numpy()

    output_offset = 2 * N * N
    
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < w_re.shape[0]:
            active_indices.append(idx)
            labels.append(val)
            
    re_weights = w_re[active_indices]
    im_weights = w_im[active_indices]
    
    joint_weights = np.concatenate([re_weights, im_weights], axis=1)

    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(joint_weights)
    
    df = pd.DataFrame(embeds_3d, columns=['PCA1', 'PCA2', 'PCA3'])
    df['label'] = labels
    
    fig = px.scatter_3d(
        df, x='PCA1', y='PCA2', z='PCA3',
        color='label', text='label',
        color_continuous_scale='RdBu_r',
        title=f"Joint 3D PCA: Real + Imaginary Weights (N={N})",
        labels={'label': 'Wartość (n)'},
        opacity=0.8
    )

    fig.update_traces(
        marker=dict(size=5, line=dict(width=1, color='Black')),
        textposition='top center'
    )

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40)
    )
    
    fig.show()

plot_joint_complex_pca_3d(model, N=6)

In [24]:
import numpy as np

def plot_input_pca_3d_polar(model, N=6, input_offset=6):
    weights = model.tokens_embed.weight.detach().cpu().numpy()
    
    input_values = list(range(-N, N + 1))
    input_indices = [val + input_offset for val in input_values]
    
    active_weights = weights[input_indices]
    
    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(active_weights)
    
    df = pd.DataFrame(embeds_3d, columns=['PCA1', 'PCA2', 'PCA3'])
    df['label'] = input_values
    
    # Obliczamy fazę i moduł dla wizualizacji (zakładając wejście jako czysto realne lub urojone)
    # Tu możesz zaszaleć i podać modelowi pary (a, b) i zobaczyć jak je układa
    
    fig = px.scatter_3d(
        df, x='PCA1', y='PCA2', z='PCA3',
        color='label', text='label',
        color_continuous_scale='Twilight'
    )
    fig.show()

In [25]:
plot_input_pca_3d_polar(model)

In [27]:
def plot_output_complex_polar_pca_3d(model, N=6):
    w_re = model.unembed_re.weight.detach().cpu().numpy()
    w_im = model.unembed_im.weight.detach().cpu().numpy()
    
    min_val, max_val = -2 * N * N, 2 * N * N
    output_offset = 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < w_re.shape[0]:
            active_indices.append(idx)
            labels.append(val)
            
    re_weights = w_re[active_indices]
    im_weights = w_im[active_indices]
    
    joint_weights = np.concatenate([re_weights, im_weights], axis=1)

    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(joint_weights)
    
    df = pd.DataFrame(embeds_3d, columns=['PCA1', 'PCA2', 'PCA3'])
    df['label'] = labels
    df['abs_label'] = np.abs(df['label'])
    
    fig = px.scatter_3d(
        df, 
        x='PCA1', 
        y='PCA2', 
        z='PCA3',
        color='label',
        text='label',
        color_continuous_scale='Picnic',
        title=f"3D PCA: Joint Output Structure (Re+Im) | N={N}",
        labels={'label': 'Wartość składowej'},
        opacity=0.8
    )

    fig.update_traces(
        marker=dict(size=5, line=dict(width=1, color='Black')),
        textposition='top center'
    )

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis_title='V1 (Global Structure)',
            yaxis_title='V2 (Periodic)',
            zaxis_title='V3 (Magnitude/Sign)'
        )
    )
    
    fig.show()

plot_output_complex_polar_pca_3d(model, N=6)


In [ ]:
def plot_output_joint_pca_2d(model, N=6):
    w_re = model.unembed_re.weight.detach().cpu().numpy()
    w_im = model.unembed_im.weight.detach().cpu().numpy()

    output_offset = 2 * N * N
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < w_re.shape[0]:
            active_indices.append(idx)
            labels.append(val)
            
    re_weights = w_re[active_indices]
    im_weights = w_im[active_indices]
    
    joint_weights = np.concatenate([re_weights, im_weights], axis=1)

    pca = PCA(n_components=2)
    embeds_2d = pca.fit_transform(joint_weights)
    
    df = pd.DataFrame(embeds_2d, columns=['PCA1', 'PCA2'])
    df['label'] = labels
    df['magnitude'] = np.abs(df['label'])

    fig = px.scatter(
        df, 
        x='PCA1', 
        y='PCA2', 
        color='label',
        text='label',
        color_continuous_scale='Picnic',
        title=f"2D PCA: Joint Output Structure (Re+Im) | N={N}",
        template="plotly_dark"
    )

    fig.update_traces(
        marker=dict(size=12, line=dict(width=1, color='white')),
        textposition='top center'
    )

    fig.show()

plot_output_joint_pca_2d(model, N=6)